In [ ]:
import pandas as pd

In [ ]:
pip install -U "datasets>=2.19" transformers torch scikit-learn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import torch
from transformers import AutoTokenizer, AutoModel

In [2]:
# =========================
# 1) Load dataset (FLAb: Jain 2024 SEC)
# =========================
URL = "https://raw.githubusercontent.com/Graylab/FLAb/main/data/aggregation/jain2024assessment_SEC.csv"
df = pd.read_csv(URL)

In [3]:
# target
y = df["SEC % Monomer"].astype(float).to_numpy()

# sanity checks
print(df.columns)
print("N =", len(df), " target mean =", y.mean(), " std =", y.std())

Index(['heavy', 'light', 'SEC % Monomer'], dtype='str')
N = 43  target mean = 97.97441860465118  std = 1.0508231597858129


In [4]:
import transformers, torch, tokenizers
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("tokenizers:", tokenizers.__version__)

transformers: 4.57.1
torch: 2.10.0+cpu
tokenizers: 0.22.2


In [11]:
import torch
print("torch:", torch.__version__, "cuda?", torch.cuda.is_available())

try:
    import torchvision
    print("torchvision:", torchvision.__version__)
    print("torchvision file:", torchvision.__file__)
except Exception as e:
    print("torchvision import error:", repr(e))

torch: 2.10.0+cpu cuda? False
torchvision import error: AttributeError("partially initialized module 'torchvision' has no attribute 'extension' (most likely due to a circular import)")


In [12]:
!pip uninstall -y torchvision torchaudio

Found existing installation: torchvision 0.22.0+cu118
Uninstalling torchvision-0.22.0+cu118:
  Successfully uninstalled torchvision-0.22.0+cu118
Found existing installation: torchaudio 2.7.0+cu118
Uninstalling torchaudio-2.7.0+cu118:
  Successfully uninstalled torchaudio-2.7.0+cu118


In [5]:
from transformers.models.esm.modeling_esm import EsmModel

In [17]:
import pandas as pd

df = pd.read_csv("your")
print("rows:", len(df), "cols:", df.shape[1])

sec_candidates = [c for c in df.columns if "sec" in c.lower()]
print("SEC candidates:", sec_candidates)

agg_candidates = [c for c in df.columns if any(k in c.lower() for k in ["aggregation", "agg", "monomer", "hmw"])]
print("Agg-related candidates (sample):", agg_candidates[:40])

rows: 246 cols: 30
SEC candidates: ['SEC %Monomer']
Agg-related candidates (sample): ['SEC %Monomer']


In [28]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr
from sklearn.preprocessing import OneHotEncoder

# =========================
# 1) Load local CSV
# =========================
CSV_PATH = "your_path" 
df = pd.read_csv(CSV_PATH)

target_col = "SEC %Monomer"
# heavy_col = "vh_protein_sequence"
# light_col = "vl_protein_sequence"

# complete protein
heavy_col = "hc_protein_sequence"
light_col = "lc_protein_sequence"

# drop rows with missing target or sequences
df = df[df[target_col].notna()].copy()
df = df[df[heavy_col].notna() & df[light_col].notna()].reset_index(drop=True)

y = df[target_col].astype(float).to_numpy()

print("N:", len(df), "target mean:", y.mean(), "std:", y.std())
print("Using:", heavy_col, "+", light_col)

# =========================
# 2) Load ESM2 (small + fast)
# =========================
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("device:", device)

# =========================
# 3) Build embeddings X (cache)
# =========================
CACHE_NPY = "GDPa1_ESM2_t6_HCLC_embeddings.npy"

@torch.no_grad()
def embed_sequence(seq: str) -> np.ndarray:
    tokens = tokenizer(seq, return_tensors="pt", truncation=True)
    tokens = {k: v.to(device) for k, v in tokens.items()}
    out = model(**tokens)
    h = out.last_hidden_state[0]  # [seq_len, dim]
    if h.shape[0] > 2:
        h = h[1:-1]
    return h.mean(dim=0).detach().cpu().numpy()

def embed_pair(h: str, l: str) -> np.ndarray:
    return np.concatenate([embed_sequence(h), embed_sequence(l)], axis=0)

if os.path.exists(CACHE_NPY):
    X = np.load(CACHE_NPY)
    print("Loaded cached embeddings:", X.shape)
else:
    X = np.vstack([embed_pair(h, l) for h, l in zip(df[heavy_col], df[light_col])])
    np.save(CACHE_NPY, X)
    print("Computed & cached embeddings:", X.shape)

# =========================
# 4) Add one-hot subtype features => X_aug
# =========================
enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")  # sklearn newer
subtype_features = enc.fit_transform(df[["hc_subtype", "lc_subtype"]])

X_aug = np.concatenate([X, subtype_features], axis=1)
print("X_aug shape:", X_aug.shape)

# =========================
# 5) CV + metrics (IMPORTANT: train on X_aug)
# =========================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
pred = np.zeros_like(y, dtype=float)

for tr, te in kf.split(X_aug):
    reg = Ridge(alpha=10.0)
    reg.fit(X_aug[tr], y[tr])
    pred[te] = reg.predict(X_aug[te])

rmse = np.sqrt(mean_squared_error(y, pred))
rho = spearmanr(y, pred).correlation
print(f"CV RMSE: {rmse:.3f}")
print(f"CV Spearman rho: {rho:.3f}")

N: 242 target mean: 95.81805785123967 std: 3.9545702910349028
Using: hc_protein_sequence + lc_protein_sequence


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


device: cpu
Computed & cached embeddings: (242, 640)
X_aug shape: (242, 645)
CV RMSE: 2.710
CV Spearman rho: 0.473


In [30]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr

fold_col = "hierarchical_cluster_IgG_isotype_stratified_fold"
folds = df[fold_col].to_numpy()

pred = np.zeros_like(y, dtype=float)

for fold in np.unique(folds):
    tr = np.where(folds != fold)[0]
    te = np.where(folds == fold)[0]

    reg = Ridge(alpha=10.0)
    reg.fit(X_aug[tr], y[tr])
    pred[te] = reg.predict(X_aug[te])

rmse = np.sqrt(mean_squared_error(y, pred))
rho = spearmanr(y, pred).correlation
print(f"Fold-CV RMSE: {rmse:.3f}")
print(f"Fold-CV Spearman rho: {rho:.3f}")

Fold-CV RMSE: 2.740
Fold-CV Spearman rho: 0.417
